In [43]:
header = ["execID", "problem", "instance", "executable", "full_instance", "status", "exit_code", "real", "time", "user", "system", "memory"]

In [44]:
import re
import os
import sys
import pandas as pd
from matplotlib import pyplot as plt

df = None
for file in os.listdir("."):
    if not file.endswith(".tsv"):
        continue
    df_curr = pd.read_csv(file, sep="\t", names=header, index_col=False)
    df = pd.concat([df, df_curr], ignore_index=True) if not df is None else df_curr

# Computing Metrics

In [45]:
# solved,par1,par10
import numpy as np


df.loc[:, "solved"] = df["status"] == "complete"
# df.loc[:, "par1"] = np.where(
#     df["status"] == "complete",
#     df["real"],
#     1200
# )
# df.loc[:, "par10"] = np.where(
#     df["status"] == "complete",
#     df["real"],
#     1200*10
# )

In [46]:
df["executable"].unique()

array(['amoclingo-lg=c-r=nomin-l=f-smpc=t',
       'amoclingo-lg=c-r=nomin-l=t-smpc=t',
       'amoclingo-lg=c-r=nomin-l=h-smpc=t',
       'amoclingo-lg=py-r=nomin-l=f-smpc=t',
       'amoclingo-lg=py-r=nomin-l=t-smpc=t',
       'amoclingo-lg=py-r=nomin-l=h-smpc=t',
       'eoclingo-lg=c-r=nomin-l=f-smpc=t',
       'eoclingo-lg=c-r=nomin-l=t-smpc=t',
       'eoclingo-lg=c-r=nomin-l=h-smpc=t',
       'eoclingo-lg=py-r=nomin-l=f-smpc=t',
       'eoclingo-lg=py-r=nomin-l=t-smpc=t',
       'eoclingo-lg=py-r=nomin-l=h-smpc=t', 'clingo-amo', 'clingo-eo',
       'wasp-amo', 'wasp-eo', 'amowasp-lg=py-r=nomin-l=f-smpc=t',
       'amowasp-lg=py-r=nomin-l=t-smpc=t',
       'amowasp-lg=py-r=nomin-l=h-smpc=t',
       'eowasp-lg=py-r=nomin-l=f-smpc=t',
       'eowasp-lg=py-r=nomin-l=t-smpc=t',
       'eowasp-lg=py-r=nomin-l=h-smpc=t', 'amoclingo-base-lg=c',
       'amoclingo-base-lg=py', 'eoclingo-base-lg=c',
       'eoclingo-base-lg=py', 'amowasp-base-lg=py', 'eowasp-base-lg=py',
       'amoclingo-

In [47]:
df[df["problem"] == "Nurse"]["executable"].unique()

array(['eoclingo-lg=c-r=nomin-l=f-smpc=t',
       'eoclingo-lg=c-r=nomin-l=t-smpc=t',
       'eoclingo-lg=c-r=nomin-l=h-smpc=t',
       'eoclingo-lg=py-r=nomin-l=f-smpc=t',
       'eoclingo-lg=py-r=nomin-l=t-smpc=t',
       'eoclingo-lg=py-r=nomin-l=h-smpc=t', 'clingo-eo', 'wasp-eo',
       'eowasp-lg=py-r=nomin-l=f-smpc=t',
       'eowasp-lg=py-r=nomin-l=t-smpc=t',
       'eowasp-lg=py-r=nomin-l=h-smpc=t', 'eoclingo-base-lg=c',
       'eoclingo-base-lg=py', 'eowasp-base-lg=py',
       'eoclingo-lg=c-r=ijcai-l=f-smpc=t',
       'eoclingo-lg=py-r=ijcai-l=f-smpc=t',
       'eowasp-lg=py-r=ijcai-l=f-smpc=t',
       'eoclingo-lg=c-r=minfly-l=f-smpc=t',
       'eoclingo-lg=c-r=minfly-l=t-smpc=t',
       'eoclingo-lg=c-r=minfly-l=h-smpc=t', 'amoclingo-base-lg=c',
       'amoclingo-base-lg=py', 'amoclingo-lg=c-r=ijcai-l=f-smpc=t',
       'amoclingo-lg=c-r=minfly-l=f-smpc=t',
       'amoclingo-lg=c-r=minfly-l=h-smpc=t',
       'amoclingo-lg=c-r=minfly-l=t-smpc=t',
       'amoclingo-lg=c-r=nomi

In [48]:
df[(df["problem"] == "Knapsack") & (df["executable"].str.contains("amoclingo-lg=c-r=nomin-l=f")) & (df["real"]>1) & (df["exit_code"] == 10)].sort_values("real")

,execID,problem,instance,executable,full_instance,status,exit_code,real,time,user,system,memory,solved
4211,firstSubmission,Knapsack,0011-knapsack-15-32223-358414-type1.asp,amoclingo-lg=c-r=nomin-l=f-smpc=t,results/Knapsack/0011-knapsack-15-32223-358414...,complete,10,10.956,10.39,10.37,0.02,53.6,True


# Df AMO

In [49]:
df_amo = df.copy()
df_amo.drop(index=df[df["executable"].str.contains(r"^eo|-eo$")].index, inplace=True)
df_amo.loc[:, "executable"] = df_amo["executable"].str.replace("-amo$","", regex=True)
df_amo["problem"].unique()

array(['GraphColouring', 'Knapsack', 'GroupAssignment', 'Nurse'],
      dtype=object)

## Pivot AMO

In [50]:
# pivot_amo = df_amo.pivot_table(index=["executable"],columns=["problem"], values=["solved","par1", "par10"], aggfunc={"solved":"sum", "par1": "mean", "par10": "mean"}, margins=True, margins_name='Total', fill_value=0)
pivot_amo = df_amo.pivot_table(index=["executable"],columns=["problem"], values=["solved"], aggfunc={"solved":"sum"}, margins=True, margins_name='Total', fill_value=-1)
pivot_amo = pivot_amo.reorder_levels([1,0], axis=1).sort_index(axis=1)
pivot_amo = pivot_amo.drop(index="Total")
# pivot_amo.loc[:, "solved"].astype(int)
for col in pivot_amo.columns[(pivot_amo.columns.get_level_values(1) == "solved")]:
    pivot_amo[col] = pivot_amo.loc[:, col].astype(int)

for col in pivot_amo.columns:
    pivot_amo[col] = pivot_amo.loc[:, col].round(2)
pivot_amo

problem,GraphColouring,GroupAssignment,Knapsack,Nurse,Total
,solved,solved,solved,solved,solved
executable,,,,,
amoclingo-base-lg=c,102,307,77,1,487
amoclingo-base-lg=py,95,141,75,0,311
amoclingo-lg=c-r=ijcai-l=f-smpc=t,120,307,77,2,506
amoclingo-lg=c-r=minfly-l=f-smpc=t,123,306,75,1,505
amoclingo-lg=c-r=minfly-l=h-smpc=t,122,310,75,0,507
amoclingo-lg=c-r=minfly-l=t-smpc=t,179,314,75,1,569
amoclingo-lg=c-r=nomin-l=f-smpc=t,122,309,76,2,509
amoclingo-lg=c-r=nomin-l=h-smpc=t,122,308,76,2,508


In [51]:
# lazy=false
# reason=nomin
# lang=cpp
# static_mpc=true

map_key_param = {
    "l": "lazy",
    "r": "reason",
    "lg": "lang",
    "l": "lazy",
    "smpc": "static_mpc"
}

map_value = {
 "t": "true",
 "f": "false",
 "h": "hybrid",
 "c": "cpp",
}

base_path = "solver_to_move_to_clown/base_amo.sh"
with open(base_path, "r") as f:
    base_sh = "".join(f.readlines())

smpc = "t"
for s in pivot_amo.index:
    s = str(s)
    if re.search("(-base-|^clingo|wasp|py)", s): continue
    content_s = str(base_sh)
    for key_value in re.findall(r"-\w+=\w+", s):
        key = re.search(r"-(?P<key>\w+)=(?P<value>\w+)", key_value).group("key")
        value = re.search(r"-(?P<key>\w+)=(?P<value>\w+)", key_value).group("value")
        content_s = re.sub(rf"{map_key_param[key]}=\w+", f"{map_key_param[key]}={map_value.get(value, value)}", content_s)
    s = f"{s}-smpc={smpc}.bash"
    print(f"Writing: {s}")
    with open(f"solver_to_move_to_clown/amoclingo_versions/{s}", "w") as f:
        print(f"# s: {s}", file=f)
        print(content_s, file=f)


Writing: amoclingo-lg=c-r=ijcai-l=f-smpc=t-smpc=t.bash
Writing: amoclingo-lg=c-r=minfly-l=f-smpc=t-smpc=t.bash
Writing: amoclingo-lg=c-r=minfly-l=h-smpc=t-smpc=t.bash
Writing: amoclingo-lg=c-r=minfly-l=t-smpc=t-smpc=t.bash
Writing: amoclingo-lg=c-r=nomin-l=f-smpc=t-smpc=t.bash
Writing: amoclingo-lg=c-r=nomin-l=h-smpc=t-smpc=t.bash
Writing: amoclingo-lg=c-r=nomin-l=t-smpc=t-smpc=t.bash


In [52]:
def create_catcus_df(df_input: pd.DataFrame) -> pd.DataFrame:
    df_catcus = pd.DataFrame()
    for solver in df_input["executable"].unique():
        df_catcus[solver] = df_input[(df_input["executable"] == solver) & (df_input["solved"])]["real"].sort_values().reset_index(drop=True)
    return df_catcus

In [53]:
df_cactus_amo = create_catcus_df(df_amo)
df_cactus_amo.to_csv("plots/catcus_amo.csv")

# Df EO

In [54]:
df_eo = df.copy()
df_eo.drop(index=df[df["executable"].str.contains("(^amo|-amo$)")].index, inplace=True)
df_eo.loc[:, "executable"] = df_eo["executable"].str.replace(r"-eo$","", regex=True)
df_eo

/var/folders/60/msx2m7995xv41v59mx28gt5w0000gn/T/ipykernel_12086/3482083397.py:2: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df_eo.drop(index=df[df["executable"].str.contains("(^amo|-amo$)")].index, inplace=True)


,execID,problem,instance,executable,full_instance,status,exit_code,real,time,user,system,memory,solved
1800,firstSubmission,GraphColouring,0001-graph_colouring-125-0_1200.asp,eoclingo-lg=c-r=nomin-l=f-smpc=t,results/GraphColouring/0001-graph_colouring-12...,complete,10,3.166,2.83,2.81,0.02,50.4,True
1801,firstSubmission,GraphColouring,0001-graph_colouring-125-0_2400.asp,eoclingo-lg=c-r=nomin-l=f-smpc=t,results/GraphColouring/0001-graph_colouring-12...,complete,10,3.647,3.33,3.30,0.03,50.3,True
1802,firstSubmission,GraphColouring,0001-graph_colouring-125-0_3600.asp,eoclingo-lg=c-r=nomin-l=f-smpc=t,results/GraphColouring/0001-graph_colouring-12...,outof time,143,1203.190,1200.79,1200.48,0.31,278.3,False
1803,firstSubmission,GraphColouring,0001-graph_colouring-125-0_4800.asp,eoclingo-lg=c-r=nomin-l=f-smpc=t,results/GraphColouring/0001-graph_colouring-12...,complete,20,560.651,558.35,558.01,0.34,418.2,True
1804,firstSubmission,GraphColouring,0001-graph_colouring-125-0_6000.asp,eoclingo-lg=c-r=nomin-l=f-smpc=t,results/GraphColouring/0001-graph_colouring-12...,complete,20,6.137,5.72,5.69,0.03,98.4,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
25645,AIJ@2026-06-04-09-49:firstSubmissionBench,GroupAssignment,345-group-assignment-250-35-middle.asp,eowasp-base-lg=py,results/GroupAssignment/345-group-assignment-2...,outof time,143,1200.881,1200.22,1195.01,5.21,2740.1,False
25646,AIJ@2026-06-04-09-49:firstSubmissionBench,GroupAssignment,346-group-assignment-250-35-middle.asp,eowasp-base-lg=py,results/GroupAssignment/346-group-assignment-2...,outof time,143,1201.541,1201.05,1197.17,3.88,2070.8,False
25647,AIJ@2026-06-04-09-49:firstSubmissionBench,GroupAssignment,347-group-assignment-250-35-middle.asp,eowasp-base-lg=py,results/GroupAssignment/347-group-assignment-2...,outof time,143,1201.562,1200.56,1195.78,4.78,2317.0,False
25648,AIJ@2026-06-04-09-49:firstSubmissionBench,GroupAssignment,348-group-assignment-250-35-punsat.asp,eowasp-base-lg=py,results/GroupAssignment/348-group-assignment-2...,outof time,143,1201.375,1200.98,1196.50,4.48,2626.3,False


In [55]:
df_cactus_eo = create_catcus_df(df_eo)
df_cactus_eo.to_csv("plots/catcus_eo.csv")

## Pivot EO

In [61]:
# pivot_eo = df_eo.pivot_table(index=["executable"],columns=["problem"], values=["solved","par1", "par10"], aggfunc={"solved":"sum", "par1": "mean", "par10": "mean"}, margins=True, margins_name='Total', fill_value=0)
pivot_eo = df_eo.pivot_table(index=["executable"],columns=["problem"], values=["solved"], aggfunc={"solved":"sum"}, margins=True, margins_name='Total', fill_value=-1)
pivot_eo = pivot_eo.reorder_levels([1,0], axis=1).sort_index(axis=1)
pivot_eo = pivot_eo.drop(index="Total")

for col in pivot_eo.columns[(pivot_eo.columns.get_level_values(1) == "solved")]:
    pivot_eo[col] = pivot_eo.loc[:, col].astype(int)
pivot_eo

problem,GraphColouring,GroupAssignment,Knapsack,Nurse,Total
,solved,solved,solved,solved,solved
executable,,,,,
clingo,95,-1,-1,4,99
eoclingo-base-lg=c,101,350,77,1,529
eoclingo-base-lg=py,79,280,74,0,433
eoclingo-lg=c-r=ijcai-l=f-smpc=t,153,-1,-1,1,154
eoclingo-lg=c-r=ijcai-l=f-smpc=t-smpc=t,-1,307,77,-1,384
eoclingo-lg=c-r=minfly-l=f-smpc=t,131,-1,-1,0,131
eoclingo-lg=c-r=minfly-l=f-smpc=t-smpc=t,-1,306,75,-1,381
eoclingo-lg=c-r=minfly-l=h-smpc=t,132,-1,-1,0,132


In [57]:
# lazy=false
# reason=nomin
# lang=cpp
# static_mpc=true

map_key_param = {
    "l": "lazy",
    "r": "reason",
    "lg": "lang",
    "l": "lazy",
    "smpc": "static_mpc"
}

map_value = {
 "t": "true",
 "f": "false",
 "h": "hybrid",
 "c": "cpp",
}

base_path = "solver_to_move_to_clown/base_eo.sh"
with open(base_path, "r") as f:
    base_sh = "".join(f.readlines())

smpc = "t"
for s in pivot_eo.index:
    s = str(s)
    if re.search("(-base-|^clingo|wasp|py)", s): continue
    content_s = str(base_sh)
    for key_value in re.findall(r"-\w+=\w+", s):
        key = re.search(r"-(?P<key>\w+)=(?P<value>\w+)", key_value).group("key")
        value = re.search(r"-(?P<key>\w+)=(?P<value>\w+)", key_value).group("value")
        content_s = re.sub(rf"{map_key_param[key]}=\w+", f"{map_key_param[key]}={map_value.get(value, value)}", content_s)
    s = f"{s}-smpc={smpc}.bash"
    print(f"Writing: {s}")
    with open(f"solver_to_move_to_clown/eoclingo_versions/{s}", "w") as f:
        print(f"# s: {s}", file=f)
        print(content_s, file=f)


Writing: eoclingo-lg=c-r=ijcai-l=f-smpc=t-smpc=t.bash
Writing: eoclingo-lg=c-r=ijcai-l=f-smpc=t-smpc=t-smpc=t.bash
Writing: eoclingo-lg=c-r=minfly-l=f-smpc=t-smpc=t.bash
Writing: eoclingo-lg=c-r=minfly-l=f-smpc=t-smpc=t-smpc=t.bash
Writing: eoclingo-lg=c-r=minfly-l=h-smpc=t-smpc=t.bash
Writing: eoclingo-lg=c-r=minfly-l=h-smpc=t-smpc=t-smpc=t.bash
Writing: eoclingo-lg=c-r=minfly-l=t-smpc=t-smpc=t.bash
Writing: eoclingo-lg=c-r=minfly-l=t-smpc=t-smpc=t-smpc=t.bash
Writing: eoclingo-lg=c-r=nomin-l=f-smpc=t-smpc=t.bash
Writing: eoclingo-lg=c-r=nomin-l=f-smpc=t-smpc=t-smpc=t.bash
Writing: eoclingo-lg=c-r=nomin-l=h-smpc=t-smpc=t.bash
Writing: eoclingo-lg=c-r=nomin-l=h-smpc=t-smpc=t-smpc=t.bash
Writing: eoclingo-lg=c-r=nomin-l=t-smpc=t-smpc=t.bash
Writing: eoclingo-lg=c-r=nomin-l=t-smpc=t-smpc=t-smpc=t.bash


# Latex Functions

In [58]:
import subprocess
from pathlib import Path
import shutil

def create_latex_visualization(latex_input: str, output_dir: str, output_name, tex_name: str = "main"):
    latex = rf"""
            \documentclass{{article}}
            \usepackage{{booktabs}}
            \usepackage{{multirow}}
            \pagestyle{{empty}}
            \begin{{document}}
            {latex_input}
            \end{{document}}
            """
    output_path = Path(f"{output_dir}/{output_name}")
    output_path.mkdir(parents=True, exist_ok=True)
    tex_name = f"{tex_name}.tex"
    tex_file = output_path / tex_name
    tex_file.write_text(latex)
    subprocess.run(
        ["/Library/TeX/texbin/pdflatex", tex_name],
        cwd=output_path
    )

## Cactus

In [59]:
def create_cactus_latex(file: str, legend_style: str = "at={(1.25,1.0)},anchor=north,fill=none", xmin=0, xmax=None, ymin=0, ymax=1200, subset_solvers = None):
    
    df_cac = pd.read_csv(file, sep=",", header="infer")
    def get_style_plot(solver: str):
        style = {}

        if re.search("amoclingo",solver):
            style["color"] = "blue"
        elif re.search("amowasp", solver):
            style["color"] = "yellow"
        elif re.search("^clingo", solver):
            style["color"] = "red"
        elif re.search("^wasp", solver):
            style["color"] = "green"
        else:
            raise Exception(f"Invalid Solver: {solver}")
        
        if re.search("amoclingo",solver):
            style["mark"] = "o"
        elif re.search("amowasp", solver):
            style["mark"] = "o"
        elif re.search("^clingo", solver):
            style["mark"] = "+"
        elif re.search("^wasp", solver):
            style["mark"] = "+"
        else:
            raise Exception(f"Invalid Solver: {solver}")
        
        return style 
    
    plot_lines = []
    columns = list(df_cac.columns)
    for i in range(1, len(columns)):
        solver = columns[i]
        if not subset_solvers is None and not solver in subset_solvers: continue
        plot_line = []
        style = get_style_plot(solver)
        plot_line = f"""
            \\addplot [mark size=2pt, color={style['color']}, mark={style['mark']}] [unbounded coords=jump] table[col sep=comma, y index={i}] {{./{file}}};
            \\addlegendentry{{{solver}}}
            """
        plot_lines.append(plot_line)

    xmax_str = f"xmax={xmax}" if xmax else ""
    tex_catctus = f"""  
    \\begin{{tikzpicture}}[scale=0.7]
        \\pgfkeys{{/pgf/number format/set thousands separator = {{}}}}
        \\begin{{axis}}[
        scale only axis
        , xlabel={{Solved instances}}
        , ylabel={{Time (s)}}    
        , xmin=0, {xmax_str}
        , ymin=0, ymax=1220
        , legend style={legend_style}
        , legend columns=1
        , width=0.65\\textwidth
        , height=0.40\\textwidth
        , major tick length=2pt
        , title= {{Catctus Plot}}
        ]

        {'\n'.join(plot_lines)}            

        \\end{{axis}}
    \\end{{tikzpicture}}%
    """
    return tex_catctus

catcus_latex_amo = create_cactus_latex("plots/catcus_amo.csv", subset_solvers=["clingo", "wasp", "amoclingo-lg=c-r=minfly-l=t"])
print(catcus_latex_amo)

  
    \begin{tikzpicture}[scale=0.7]
        \pgfkeys{/pgf/number format/set thousands separator = {}}
        \begin{axis}[
        scale only axis
        , xlabel={Solved instances}
        , ylabel={Time (s)}    
        , xmin=0, 
        , ymin=0, ymax=1220
        , legend style=at={(1.25,1.0)},anchor=north,fill=none
        , legend columns=1
        , width=0.65\textwidth
        , height=0.40\textwidth
        , major tick length=2pt
        , title= {Catctus Plot}
        ]

        
            \addplot [mark size=2pt, color=red, mark=+] [unbounded coords=jump] table[col sep=comma, y index=7] {./plots/catcus_amo.csv};
            \addlegendentry{clingo}
            

            \addplot [mark size=2pt, color=green, mark=+] [unbounded coords=jump] table[col sep=comma, y index=8] {./plots/catcus_amo.csv};
            \addlegendentry{wasp}
                        

        \end{axis}
    \end{tikzpicture}%
    


## Tables

In [60]:
def get_pivot_latex(pivot: pd.DataFrame):
    pivot_latex = pivot.copy()

    for c in pivot_latex.columns:
        maxVal = pivot_latex[c].max()
        pivot_latex[c] = pivot_latex[c].astype(object)
        for r in pivot_latex.index:
            v = pivot_latex.at[r, c]
            if v == maxVal:
                pivot_latex.at[r, c] = rf"\textbf{{{v}}}"
            else:
                pivot_latex.at[r, c] = str(v)
    display(pivot_latex)
    latex = pivot_latex.to_latex(
        multicolumn=True,           
        multicolumn_format='c',     
        multirow=True,              
        bold_rows=False,    
        na_rep='-',                 
        label="tab:results",
        position="t!",
        escape=False,               
    )
    return latex

table_amo_res = get_pivot_latex(pivot_amo)
print(table_amo_res)
create_latex_visualization(table_amo_res, "tables", "table_amo")

problem,GraphColouring,GroupAssignment,Knapsack,Nurse,Total
,solved,solved,solved,solved,solved
executable,,,,,
amoclingo-base-lg=c,102,307,\textbf{77},1,487
amoclingo-base-lg=py,95,141,75,0,311
amoclingo-lg=c-r=ijcai-l=f-smpc=t,120,307,\textbf{77},2,506
amoclingo-lg=c-r=minfly-l=f-smpc=t,123,306,75,1,505
amoclingo-lg=c-r=minfly-l=h-smpc=t,122,310,75,0,507
amoclingo-lg=c-r=minfly-l=t-smpc=t,\textbf{179},\textbf{314},75,1,\textbf{569}
amoclingo-lg=c-r=nomin-l=f-smpc=t,122,309,76,2,509
amoclingo-lg=c-r=nomin-l=h-smpc=t,122,308,76,2,508


\begin{table}[t!]
\label{tab:results}
\begin{tabular}{llllll}
\toprule
problem & GraphColouring & GroupAssignment & Knapsack & Nurse & Total \\
 & solved & solved & solved & solved & solved \\
executable &  &  &  &  &  \\
\midrule
amoclingo-base-lg=c & 102 & 307 & \textbf{77} & 1 & 487 \\
amoclingo-base-lg=py & 95 & 141 & 75 & 0 & 311 \\
amoclingo-lg=c-r=ijcai-l=f-smpc=t & 120 & 307 & \textbf{77} & 2 & 506 \\
amoclingo-lg=c-r=minfly-l=f-smpc=t & 123 & 306 & 75 & 1 & 505 \\
amoclingo-lg=c-r=minfly-l=h-smpc=t & 122 & 310 & 75 & 0 & 507 \\
amoclingo-lg=c-r=minfly-l=t-smpc=t & \textbf{179} & \textbf{314} & 75 & 1 & \textbf{569} \\
amoclingo-lg=c-r=nomin-l=f-smpc=t & 122 & 309 & 76 & 2 & 509 \\
amoclingo-lg=c-r=nomin-l=h-smpc=t & 122 & 308 & 76 & 2 & 508 \\
amoclingo-lg=c-r=nomin-l=t-smpc=t & 165 & 313 & \textbf{77} & \textbf{3} & 558 \\
amoclingo-lg=py-r=ijcai-l=f-smpc=t & 118 & 229 & 76 & -1 & 423 \\
amoclingo-lg=py-r=nomin-l=f-smpc=t & 115 & 230 & 75 & -1 & 420 \\
amoclingo-lg=py-r=nomin